# P0 Token Correction: Regret Threshold Sweep (0.1–0.8)

固定 `threshold=0.4`、`correction=regret`、`scope=global`，扫描 **regret_threshold**（反悔阈值）从 0.1 到 0.8。

**核心设计：任务池 + GPU 池**
- 将所有 `regret_threshold` 配置排入任务队列
- 可用 GPU 组成 GPU 池
- 每个 GPU 从任务池中取一个任务执行，完成后自动取下一个
- 适用于 GPU 数量少于任务数量的场景（10 个任务，8 个 GPU）

**核心指标：**
- Flexible ACC / Strict ACC vs regret_threshold
- Total NFE vs regret_threshold
- Avg Corrections vs regret_threshold

**总计：2 baseline + 8 regret_threshold 扫描 = 10 组配置**

## 1. 环境设置

In [ ]:
import os
import torch
import gc

# Set GPU (modify as needed)
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'

# Environment settings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

# Change to llada directory
os.chdir('llada')

# Create log directory
os.makedirs('nlogs', exist_ok=True)

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")

## 2. 扫参配置 & 任务池 / GPU 池

- `REGRET_THRESHOLDS`: 0.1 ~ 0.8，步长 0.1（共 8 个值）
- 另加 2 个 baseline：`t0.9_none`（保守）、`t0.4_none`（激进无纠错）
- `GPU_POOL`: 可用的 GPU 编号列表
- 调度逻辑：每个 GPU 从任务队列取任务，跑完一个再取下一个，直到队列为空

In [ ]:
import subprocess
import datetime
import threading
import queue
import numpy as np

# ========== 扫参配置 ==========
REGRET_THRESHOLDS = [round(0.1 * i, 1) for i in range(1, 9)]  # 0.1 ~ 0.8

# ========== GPU 池 ==========
GPU_POOL = [0, 1, 2, 3, 4, 5, 6, 7]

# ========== 固定参数 ==========
LIMIT = 200            # 样本数（0 = 全量 GSM8K 1319 题）
GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.4        # 固定 unmask 阈值

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = f"correction_sweep_results/{timestamp}"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ========== 构建任务列表 ==========
TASKS = []

# Baseline 1: t0.9 none
TASKS.append({
    'config_name': 't0.9_none',
    'threshold': 0.9,
    'correction': 'none',
    'scope': 'block',
    'regret_threshold': 0.1,
})

# Baseline 2: t0.4 none
TASKS.append({
    'config_name': 't0.4_none',
    'threshold': 0.4,
    'correction': 'none',
    'scope': 'block',
    'regret_threshold': 0.1,
})

# Sweep: regret_threshold 0.1 ~ 0.8
for rt in REGRET_THRESHOLDS:
    TASKS.append({
        'config_name': f't0.4_regret_glb_rt{rt}',
        'threshold': 0.4,
        'correction': 'regret',
        'scope': 'global',
        'regret_threshold': rt,
    })

print(f"Sweep regret_threshold: {REGRET_THRESHOLDS}")
print(f"Total tasks: {len(TASKS)}")
print(f"GPU pool: {GPU_POOL} ({len(GPU_POOL)} GPUs)")
print(f"LIMIT: {LIMIT}")
print(f"Timestamp: {timestamp}")
print(f"Results dir: {RESULTS_DIR}")
print()
for i, t in enumerate(TASKS):
    print(f"  [{i:2d}] {t['config_name']:30s} threshold={t['threshold']} regret_threshold={t['regret_threshold']}")

## 3. 任务池调度：启动所有实验

In [ ]:
task_queue = queue.Queue()
for cfg in TASKS:
    task_queue.put(cfg)

results_lock = threading.Lock()
all_run_results = []  # (config_name, log_file, output_path, return_code)


def gpu_worker(gpu_id):
    """GPU worker: 不断从任务队列取任务，直到队列为空"""
    while True:
        try:
            cfg = task_queue.get_nowait()
        except queue.Empty:
            return

        name = cfg['config_name']
        log_file = f"nlogs/p0corr_{name}_{timestamp}.log"
        output_path = f"{RESULTS_DIR}/{name}.pkl"

        cmd = (
            f"CUDA_VISIBLE_DEVICES={gpu_id} python _p0_runner.py "
            f"--config_name {name} "
            f"--threshold {cfg['threshold']} "
            f"--correction {cfg['correction']} "
            f"--scope {cfg['scope']} "
            f"--regret_threshold {cfg['regret_threshold']} "
            f"--limit {LIMIT} "
            f"--gen_length {GEN_LENGTH} "
            f"--steps {STEPS} "
            f"--block_length {BLOCK_LENGTH} "
            f"--output_path {output_path}"
        )

        print(f"[GPU {gpu_id}] START  {name}")

        p = subprocess.Popen(
            cmd, shell=True,
            stdout=open(log_file, 'w'),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()

        status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
        print(f"[GPU {gpu_id}] DONE   {name}  {status}")

        with results_lock:
            all_run_results.append((name, log_file, output_path, rc))

        task_queue.task_done()


# 启动 GPU worker 线程
threads = []
for gpu_id in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True)
    t.start()
    threads.append(t)

print(f"\nLaunched {len(threads)} GPU workers for {len(TASKS)} tasks.")
print("Waiting for all tasks to complete...")

In [ ]:
# 等待所有线程完成
for t in threads:
    t.join()

print(f"\nAll {len(all_run_results)} / {len(TASKS)} tasks finished.")

all_run_results.sort(key=lambda x: x[0])

for name, log_file, output_path, rc in all_run_results:
    status = 'OK' if rc == 0 else f'FAILED(exit={rc})'
    print(f"  {name:30s}  {status}  log={log_file}")

## 4. 解析评测结果（从 pickle + log 文件）

In [ ]:
import pickle
import re
import pandas as pd

all_results = {}  # config_name -> data dict

for cfg in TASKS:
    name = cfg['config_name']
    pkl_path = f"{RESULTS_DIR}/{name}.pkl"
    if not os.path.exists(pkl_path):
        print(f"[MISSING] {name}")
        continue
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    all_results[name] = data

print(f"Loaded {len(all_results)} / {len(TASKS)} results.")
print()

## 5. 汇总表格（Strict + Flexible ACC）

In [ ]:
def parse_regret_threshold(config_name: str) -> str:
    m = re.search(r'_rt([\d.]+)$', config_name)
    return m.group(1) if m else '—'

rows = []
for name, data in all_results.items():
    rows.append({
        'Config': name,
        'Regret Thresh': parse_regret_threshold(name),
        'Flex ACC': f"{data['flex_acc']:.2%}",
        'Flex SE': f"{data['flex_se']:.4f}",
        'Strict ACC': f"{data['strict_acc']:.2%}",
        'Strict SE': f"{data['strict_se']:.4f}",
        'Avg NFE': f"{data['avg_nfe']:.1f}",
        'Avg Corr': f"{data['avg_corrections']:.1f}",
        'Tok/s': f"{data['tokens_per_second']:.1f}",
        'Time(s)': f"{data['elapsed']:.0f}",
        'Samples': data['n_samples'],
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
# lm_eval style table
w = max(len(n) for n in all_results) + 2
print(f'|{"Tasks":<{w}}|Version|     Filter     |n-shot|  Metric   |   | Value|   |Stderr|')
print(f'|{"-"*w}|------:|----------------|-----:|-----------|---|-----:|---|-----:|')
for name, data in all_results.items():
    print(f'|{name:<{w}}|      3|flexible-extract|     5|exact_match|↑  |{data["flex_acc"]:5.2f}|±  |{data["flex_se"]:.4f}|')
    print(f'|{"":<{w}}|       |strict-match    |     5|exact_match|↑  |{data["strict_acc"]:5.2f}|±  |{data["strict_se"]:.4f}|')

## 6. 可视化：Regret Threshold Sweep

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11})

# Extract sweep data
sweep_rts, sweep_flex, sweep_strict, sweep_nfe, sweep_corr = [], [], [], [], []

for name, data in all_results.items():
    m = re.search(r'_rt([\d.]+)$', name)
    if m:
        sweep_rts.append(float(m.group(1)))
        sweep_flex.append(data['flex_acc'] * 100)
        sweep_strict.append(data['strict_acc'] * 100)
        sweep_nfe.append(data['avg_nfe'])
        sweep_corr.append(data['avg_corrections'])

# Sort by regret_threshold
order = np.argsort(sweep_rts)
sweep_rts    = [sweep_rts[i] for i in order]
sweep_flex   = [sweep_flex[i] for i in order]
sweep_strict = [sweep_strict[i] for i in order]
sweep_nfe    = [sweep_nfe[i] for i in order]
sweep_corr   = [sweep_corr[i] for i in order]

# Baselines
bl = {}
for bname in ['t0.9_none', 't0.4_none']:
    if bname in all_results:
        bl[bname] = all_results[bname]

bl_colors = {'t0.9_none': '#FF5722', 't0.4_none': '#2196F3'}
bl_labels = {'t0.9_none': 'Baseline t=0.9 (no corr)', 't0.4_none': 'Baseline t=0.4 (no corr)'}

print(f'Sweep points: {len(sweep_rts)}')
print(f'Baselines: {list(bl.keys())}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (a) Flexible ACC vs regret_threshold
ax = axes[0]
ax.plot(sweep_rts, sweep_flex, 'o-', color='#4CAF50', linewidth=2, markersize=7, label='Regret Global')
for bname, bdata in bl.items():
    ax.axhline(bdata['flex_acc'] * 100, color=bl_colors[bname], linestyle='--', linewidth=1.5, label=bl_labels[bname])
ax.set_xlabel('Regret Threshold')
ax.set_ylabel('Flexible ACC (%)')
ax.set_title('(a) Flexible ACC vs Regret Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
for x, y in zip(sweep_rts, sweep_flex):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

# (b) Avg NFE vs regret_threshold
ax = axes[1]
ax.plot(sweep_rts, sweep_nfe, 's-', color='#9C27B0', linewidth=2, markersize=7, label='Regret Global')
for bname, bdata in bl.items():
    ax.axhline(bdata['avg_nfe'], color=bl_colors[bname], linestyle='--', linewidth=1.5, label=bl_labels[bname])
ax.set_xlabel('Regret Threshold')
ax.set_ylabel('Avg NFE')
ax.set_title('(b) Average NFE vs Regret Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
for x, y in zip(sweep_rts, sweep_nfe):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

# (c) Avg Corrections vs regret_threshold
ax = axes[2]
ax.plot(sweep_rts, sweep_corr, 'D-', color='#E040FB', linewidth=2, markersize=7, label='Regret Global')
ax.set_xlabel('Regret Threshold')
ax.set_ylabel('Avg Corrections')
ax.set_title('(c) Avg Corrections vs Regret Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
for x, y in zip(sweep_rts, sweep_corr):
    ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

fig.suptitle('Regret Threshold Sweep (threshold=0.4, regret, global)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../correction_analysis', exist_ok=True)
plt.savefig('../correction_analysis/p0_regret_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correction_analysis/p0_regret_sweep.png')

## 7. 对比分析：不同 regret_threshold 的 fix/break

In [ ]:
baseline_04 = all_results.get('t0.4_none')
if baseline_04 is None:
    print('WARNING: t0.4_none baseline not found, skipping comparison.')
else:
    base_flex = baseline_04['flex_acc'] * 100
    print(f'Baseline (t0.4_none): Flex ACC = {base_flex:.1f}%')
    print()
    print(f'{"Config":>30s} {"Flex ACC":>10s} {"Δ vs base":>10s} {"Avg NFE":>10s} {"Avg Corr":>10s}')
    print('-' * 75)

    for name in all_results:
        data = all_results[name]
        flex = data['flex_acc'] * 100
        delta = flex - base_flex
        delta_str = f'{delta:+.1f}pp' if name != 't0.4_none' else '—'
        print(f'{name:>30s} {flex:>9.1f}% {delta_str:>10s} {data["avg_nfe"]:>10.1f} {data["avg_corrections"]:>10.1f}')

In [ ]:
# Per-sample fix / break analysis vs t0.4_none baseline
if baseline_04 is not None:
    print('=== Per-sample fix/break vs t0.4_none (flexible acc) ===')
    print(f'{"Config":>30s} {"Fixed":>8s} {"Broke":>8s} {"Net":>8s}')
    print('-' * 60)

    for name, data in all_results.items():
        if name in ('t0.9_none', 't0.4_none'):
            continue

        fixed, broke = 0, 0
        n = min(len(baseline_04['results']), len(data['results']))
        for i in range(n):
            b = baseline_04['results'][i]['flex_ok']
            c = data['results'][i]['flex_ok']
            if not b and c:
                fixed += 1
            elif b and not c:
                broke += 1

        net = fixed - broke
        print(f'{name:>30s} {f"+{fixed}":>8s} {f"-{broke}":>8s} {f"{net:+d}":>8s}')

## 8. ACC vs NFE trade-off 散点图

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Sweep points
for name, data in all_results.items():
    m = re.search(r'_rt([\d.]+)$', name)
    if m:
        rt = float(m.group(1))
        ax.scatter(data['avg_nfe'], data['flex_acc'] * 100,
                   c='#4CAF50', marker='D', s=100, alpha=0.85,
                   edgecolors='k', linewidths=0.5, zorder=3)
        ax.annotate(f'rt={rt}', (data['avg_nfe'], data['flex_acc'] * 100),
                    textcoords='offset points', xytext=(6, 6), fontsize=8)

# Baselines
for bname, bdata in bl.items():
    color = bl_colors.get(bname, '#333')
    ax.scatter(bdata['avg_nfe'], bdata['flex_acc'] * 100,
               c=color, marker='*', s=200, edgecolors='k', linewidths=0.5, zorder=4)
    ax.annotate(bname, (bdata['avg_nfe'], bdata['flex_acc'] * 100),
                textcoords='offset points', xytext=(6, -10), fontsize=9, fontweight='bold')

# Legend
ax.scatter([], [], c='#4CAF50', marker='D', s=100, label='Regret Global (sweep)', edgecolors='k')
ax.scatter([], [], c='#FF5722', marker='*', s=200, label='Baseline t=0.9', edgecolors='k')
ax.scatter([], [], c='#2196F3', marker='*', s=200, label='Baseline t=0.4', edgecolors='k')

ax.set_xlabel('Average NFE (lower = faster)')
ax.set_ylabel('Flexible ACC (%)')
ax.set_title('ACC vs NFE Trade-off: Regret Threshold Sweep')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../correction_analysis/p0_regret_acc_nfe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: correction_analysis/p0_regret_acc_nfe.png')